# NB7b -- Defence 2 Hardened: Cyber-Weighted, K-Fold Ensemble Gate (Multiclass, v4)

Hardens the single-AE gate from `NB7_Defence2_MultiClass_v4` along the three
axes identified after interpreting its results: high test-set FPR (15.7% vs.
a 5% calibration target), high seed-to-seed variance, and a flat reconstruction
error signal diluted across 33 physics features the attacker never touches.

**Three fixes, combined into one coherent design:**

1. **Cyber-weighted reconstruction error.** PPO/adaptive attacks only ever
   perturb the 10 attacker-controllable cyber features -- averaging error
   equally over all 43 features dilutes a real perturbation signal in those
   10 with noise from 33 untouched physics features. A per-feature weight
   vector boosts the 10 cyber features (`CYBER_BOOST`), used in **both** the
   AE's training loss and its scoring function, so the network actually
   allocates bottleneck capacity toward representing the cyber-feature
   manifold accurately.

2. **Ensemble of gates** and **3. better threshold calibration** turn out to
   be the same fix: 5-fold cross-validation over the benign training pool
   naturally gives (a) 5 fold-models to ensemble (mean score across all 5 at
   inference) and (b) an honest, non-overfit threshold -- each benign training
   sample's out-of-fold error comes from a model that never saw it during
   training, pooled across all 5 folds and thresholded at the 95th percentile.
   This is more representative than NB7's single 80/20 validation split.

**One calibration/deployment subtlety, stated explicitly rather than glossed
over**: the threshold is calibrated from each sample's *single* excluded-fold
model score (standard k-fold OOF practice), but deployment scoring uses the
*mean across all 5* fold-models. Ensemble averaging typically reduces error
variance relative to any single model, so a threshold calibrated on
(noisier) single-model OOF scores is a reasonably conservative choice for the
(smoother) ensemble-mean score used at inference -- not a perfect match, but
a defensible one, flagged here rather than presented as airtight.

**Compared directly against NB7's saved baseline** (`defence2_results_v4.pkl`)
at the end, so the hardening's actual effect is quantified, not assumed.


---

**This revision (Defence 2 Hardened, baseline-model variant):** the original NB7b gates the Optuna-tuned XGBoost's defended model, matching Attack 1/2/3 and Defence 1's original target. This revision retargets the identical hardened-gate methodology (cyber-weighted k-fold ensemble, threshold calibration, 3-build reproducibility check) at the untuned baseline pipeline instead. Seven cross-referenced files were fixed: the Standard model now uses fixed baseline hyperparameters instead of loading `optuna_xgboost_results_v4.pkl`; the defended model, PGD adversarial examples, Attack 3 comparison data, adaptive surrogate, and both PPO agents (unconstrained and budget-matched) all load their `*_baseline_v4` equivalents. Output filenames are suffixed `_baseline_v4` throughout. **Two comparisons in this notebook (cell 6's NB7-vs-NB7b table, and cell 9's 0.2142 reproducibility check) load the ORIGINAL tuned-model single-AE gate's results as their comparison point, since no baseline-model equivalent of that simpler gate exists** -- both are flagged inline as reference-only, since they now mix a gate-architecture change with a classifier change rather than isolating hardening's effect alone. **Dependency: requires `xgb_defended_model_baseline_v4.json`, `X_adv_pgd_eps01_baseline_v4.npy`, `attack3_results_baseline_v4.pkl`, `adaptive_surrogate_baseline_v4.pt`, `ppo_evasion_agent_unconstrained_baseline_v4.zip`, and `ppo_evasion_agent_budgeted_baseline_v4.zip` to already exist on disk** -- i.e. Attack 1, Attack 2, Attack 3, and Defence 1's baseline notebooks must all be run first.


## Setup

In [ ]:
!pip install -q sb3-contrib gymnasium


In [ ]:
import os, random, warnings, copy
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
import xgboost as xgb
from sklearn.metrics import f1_score
import gymnasium as gym
from gymnasium import spaces
from sb3_contrib import RecurrentPPO
warnings.filterwarnings('ignore')

# Drive-ready: PHYSAR-NEW lives on Google Drive (survives Colab runtime restarts)
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/PHYSAR-NEW'
os.chdir(PROJECT_DIR)
if not os.path.exists('phase1_data_v4.pkl'):
    print(f'WARNING: phase1_data_v4.pkl not found in {PROJECT_DIR} -- fix PROJECT_DIR before continuing.')
    print(sorted(os.listdir(PROJECT_DIR))[:20])
else:
    print('Found phase1_data_v4.pkl -- PROJECT_DIR looks correct.')

RANDOM_STATE = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Working directory: {os.getcwd()}')
print(f'Device: {DEVICE}')

with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _test_model = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2)
    _test_model.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _fell_back_to_cpu = any('No visible GPU' in str(w.message) for w in _caught)

if _fell_back_to_cpu:
    GPU_PARAMS = {}
    print('GPU not available for XGBoost -- using CPU.')
else:
    GPU_PARAMS = {'tree_method': 'hist', 'device': 'cuda'}
    print('GPU detected -- XGBoost will use CUDA.')


## 1. Load v4 artifacts (same as NB7)

In [ ]:
PIPELINE_VERSION = 'v4'
EXPECTED_TEST_FRAC = 0.15
EXPECTED_EMBARGO_SECONDS = 1.0

data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
version_mismatch = (
    data.get('PIPELINE_VERSION') != PIPELINE_VERSION
    or data.get('TEST_FRAC') != EXPECTED_TEST_FRAC
    or data.get('EMBARGO_SECONDS') != EXPECTED_EMBARGO_SECONDS
)
if version_mismatch:
    print('WARNING: version metadata mismatch -- proceeding anyway, double check first.')
else:
    print(f'Version check passed: {PIPELINE_VERSION}')

X_train_scaled = data['X_train_scaled']
X_test_scaled  = data['X_test_scaled']
y_train = pd.Series(data['y_train'])
y_test  = pd.Series(data['y_test'])
FEATURE_COLS = data['FEATURE_COLS']
y_train_np = y_train.values
y_test_np  = y_test.values
n_feat = X_train_scaled.shape[1]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
attack_mask = (y_test_np != 0)
benign_mask = (y_test_np == 0)

# CHANGED: this notebook now targets the UNTUNED baseline XGBoost (Table 6.1's
# headline model, F1=0.7225) rather than the Optuna-tuned configuration (F1=0.7110)
# the original NB7b used -- companion to the Attack 1/2/3 and Defence 1 baseline checks.
baseline_results = joblib.load(f'baseline_results_{PIPELINE_VERSION}.pkl')
best_params = dict(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', num_class=6, eval_metric='mlogloss',
    random_state=RANDOM_STATE, n_jobs=-1, **GPU_PARAMS,
)
xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(X_train_scaled, y_train, verbose=False)
standard_test_f1 = f1_score(y_test, xgb_model.predict(X_test_scaled), average='weighted')
print(f'Standard XGBoost (untuned baseline) -- Test F1: {standard_test_f1:.4f} '
      f'(recorded: {baseline_results["xgb_test_f1"]:.4f})')

defended_booster = xgb.Booster()
defended_booster.load_model(f'xgb_defended_model_baseline_{PIPELINE_VERSION}.json')

class DefendedModelWrapper:
    def predict(self, X):
        probs = defended_booster.predict(xgb.DMatrix(X))
        return probs.argmax(axis=1)
    def predict_proba(self, X):
        return defended_booster.predict(xgb.DMatrix(X))

xgb_defended = DefendedModelWrapper()
defended_test_f1 = f1_score(y_test, xgb_defended.predict(X_test_scaled), average='weighted')
print(f'Defended XGBoost (loaded) -- Test F1: {defended_test_f1:.4f}')

# CHANGED: cross-reference the baseline-model Attack 1/Attack 3 runs.
X_adv_pgd = np.load(f'X_adv_pgd_eps01_baseline_{PIPELINE_VERSION}.npy')
attack3_data = joblib.load(f'attack3_results_baseline_{PIPELINE_VERSION}.pkl')

CYBER_FEATURE_NAMES = [
    'cyber_pkt_count', 'cyber_pktlen_mean', 'cyber_pktlen_std',
    'cyber_n_unique_srcport', 'cyber_n_unique_dstport',
    'cyber_modbus_frac', 'cyber_tls_frac', 'cyber_tcp_frac',
    'cyber_iat_mean', 'cyber_iat_std',
]
CYBER_IDX = np.array([FEATURE_COLS.index(c) for c in CYBER_FEATURE_NAMES])
N_FEATURES = len(FEATURE_COLS)
print(f'Cyber (attacker-controllable) feature indices: {CYBER_IDX.tolist()}')


## 2. Pre-generate adversarial example sets (once, same as NB7)

In [ ]:
pgd_evasion_standard = (xgb_model.predict(X_adv_pgd)[attack_mask] == 0).mean()
pgd_evasion_defended = (xgb_defended.predict(X_adv_pgd)[attack_mask] == 0).mean()
print(f'PGD evasion (no gate) -- Standard: {pgd_evasion_standard:.4f}  Defended: {pgd_evasion_defended:.4f}')

class AdaptiveSurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, n_classes),
        )
    def forward(self, x):
        return self.net(x)

adaptive_surrogate = AdaptiveSurrogateMLP(n_feat).to(DEVICE)
adaptive_surrogate.load_state_dict(torch.load(f'adaptive_surrogate_baseline_{PIPELINE_VERSION}.pt', map_location=DEVICE))
adaptive_surrogate.eval()

def pgd_adaptive(model, X, y_np, epsilon=0.1, alpha=None, n_iter=10):
    model.eval()
    if alpha is None:
        alpha = epsilon / 4
    y_t = torch.from_numpy(y_np).long().to(DEVICE)
    X_orig = torch.from_numpy(X).float().to(DEVICE)
    X_adv = X_orig.clone().detach()
    for _ in range(n_iter):
        X_adv.requires_grad_(True)
        loss = F.cross_entropy(model(X_adv), y_t)
        grad = torch.autograd.grad(loss, X_adv)[0]
        X_adv = X_adv.detach() + alpha * grad.sign()
        X_adv = (X_orig + torch.clamp(X_adv - X_orig, -epsilon, epsilon)).detach()
    return X_adv.cpu().numpy()

BATCH = 4096
X_adv_adaptive = np.zeros_like(X_test_scaled)
for start in range(0, len(X_test_scaled), BATCH):
    end = min(start + BATCH, len(X_test_scaled))
    X_adv_adaptive[start:end] = pgd_adaptive(adaptive_surrogate, X_test_scaled[start:end], y_test_np[start:end])
adaptive_evasion_defended = (xgb_defended.predict(X_adv_adaptive)[attack_mask] == 0).mean()
print(f'Adaptive PGD evasion (no gate, defended, this surrogate): {adaptive_evasion_defended:.4f}')

EPS_BOUND = 0.3
MAX_STEPS = 10

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool, y_pool, model, cyber_idx, n_features, n_classes=6,
                 eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None, eps_total_budget=None):
        super().__init__()
        self.X_pool = X_pool; self.y_pool = y_pool; self.model = model
        self.cyber_idx = np.array(cyber_idx); self.n_features = n_features
        self.n_cyber = len(cyber_idx); self.n_classes = n_classes
        self.eps_bound = eps_bound; self.max_steps = max_steps
        self.eps_total_budget = eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(self.n_cyber,), dtype=np.float32)
        obs_dim = self.n_features + self.n_cyber + self.n_classes + 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

    def _get_obs(self):
        oh = np.zeros(self.n_classes, dtype=np.float32); oh[self.last_pred] = 1.0
        return np.concatenate([self.x_current.astype(np.float32), self.last_perturbation.astype(np.float32),
                                oh, [self.step_count / self.max_steps]])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy(); self.x_current = self.x_orig.copy()
        self.y_true = self.y_pool[idx]; self.step_count = 0
        self.last_perturbation = np.zeros(self.n_cyber, dtype=np.float32)
        self.last_pred = self.model.predict(self.x_current.reshape(1, -1))[0]
        return self._get_obs(), {}

    def step(self, action):
        action = np.clip(action, -self.eps_bound, self.eps_bound)
        if self.eps_total_budget is not None:
            proposed = (self.x_current[self.cyber_idx] + action) - self.x_orig[self.cyber_idx]
            proposed = np.clip(proposed, -self.eps_total_budget, self.eps_total_budget)
            self.x_current[self.cyber_idx] = self.x_orig[self.cyber_idx] + proposed
        else:
            self.x_current[self.cyber_idx] = self.x_current[self.cyber_idx] + action
        self.last_perturbation = action.astype(np.float32); self.step_count += 1
        pred = self.model.predict(self.x_current.reshape(1, -1))[0]; self.last_pred = pred
        terminated = truncated = False
        if pred == 0:
            reward = 10.0 - 0.01 * np.sum(action ** 2); terminated = True
        elif self.step_count >= self.max_steps:
            reward = -1.0 - 0.01 * np.sum(action ** 2); truncated = True
        else:
            reward = -0.01 * np.sum(action ** 2)
        return self._get_obs(), reward, terminated, truncated, {'evaded': pred == 0, 'x_current': self.x_current.copy()}

attack_test_mask = (y_test_np != 0)
X_attack_test = X_test_scaled[attack_test_mask]
y_attack_test = y_test_np[attack_test_mask]

env_for_loading = IDSEvasionEnv(X_pool=X_attack_test, y_pool=y_attack_test, model=xgb_defended,
                                 cyber_idx=CYBER_IDX, n_features=N_FEATURES, seed=RANDOM_STATE)
# CHANGED: load Attack 2's BASELINE-model PPO agents.
ppo_model_unconstrained = RecurrentPPO.load('ppo_evasion_agent_unconstrained_baseline_v4', env=env_for_loading)
ppo_model_budgeted = RecurrentPPO.load('ppo_evasion_agent_budgeted_baseline_v4', env=env_for_loading)
print('Both PPO agents loaded (not retrained).')

N_PER_CLASS = 500

def collect_ppo_evasion(ppo_agent, target_model, eps_total_budget, n_per_class=N_PER_CLASS, seed=RANDOM_STATE):
    X_evaded, y_evaded, evaded_flags = [], [], []
    for cls_idx in range(1, 6):
        cls_mask = np.where(y_attack_test == cls_idx)[0]
        chosen = np.random.RandomState(seed).choice(cls_mask, size=min(n_per_class, len(cls_mask)), replace=False)
        for i in chosen:
            env_tmp = IDSEvasionEnv(X_pool=X_attack_test[i:i+1], y_pool=y_attack_test[i:i+1], model=target_model,
                                     cyber_idx=CYBER_IDX, n_features=N_FEATURES, seed=0, eps_total_budget=eps_total_budget)
            obs, _ = env_tmp.reset()
            lstm_states, episode_starts = None, np.ones((1,), dtype=bool)
            x_final = X_attack_test[i].copy(); evaded_this = False
            for _ in range(MAX_STEPS):
                action, lstm_states = ppo_agent.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
                episode_starts = np.zeros((1,), dtype=bool)
                obs, _, terminated, truncated, info = env_tmp.step(action)
                x_final = info['x_current']
                if info.get('evaded', False):
                    evaded_this = True
                if terminated or truncated:
                    break
            X_evaded.append(x_final); y_evaded.append(cls_idx); evaded_flags.append(evaded_this)
    return np.array(X_evaded, dtype=np.float32), np.array(y_evaded), np.array(evaded_flags)

print('Collecting PPO evasion samples (unconstrained agent vs. defended model)...')
X_ppo_unc, y_ppo_unc, evaded_ppo_unc = collect_ppo_evasion(ppo_model_unconstrained, xgb_defended, eps_total_budget=None)
print(f'  Actual evasion rate (no gate): {evaded_ppo_unc.mean():.4f}')

print('Collecting PPO evasion samples (budget-matched agent vs. defended model)...')
X_ppo_bud, y_ppo_bud, evaded_ppo_bud = collect_ppo_evasion(ppo_model_budgeted, xgb_defended, eps_total_budget=0.1)
print(f'  Actual evasion rate (no gate): {evaded_ppo_bud.mean():.4f}')


## 3. Hardened gate: cyber-weighted autoencoder + k-fold ensemble/calibration

In [ ]:
CYBER_BOOST = 4.0  # per-feature weight multiplier for the 10 cyber features

feature_weights = np.ones(N_FEATURES, dtype=np.float32)
feature_weights[CYBER_IDX] = CYBER_BOOST
feature_weights_t = torch.from_numpy(feature_weights).to(DEVICE)
print(f'Cyber features weighted {CYBER_BOOST}x in both training loss and scoring '
      f'({len(CYBER_IDX)}/{N_FEATURES} features carry '
      f'{feature_weights.sum()/N_FEATURES*100/(feature_weights.mean()):.0f}% relative influence).')

class AnomalyAutoencoder(nn.Module):
    def __init__(self, n_features, bottleneck=8, hidden=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, hidden), nn.GELU(),
            nn.Linear(hidden, bottleneck), nn.GELU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, hidden), nn.GELU(),
            nn.Linear(hidden, n_features),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

def weighted_mse(recon, target):
    sq_err = (recon - target) ** 2
    return (sq_err * feature_weights_t).sum(dim=1) / feature_weights_t.sum()

def weighted_reconstruction_errors(model, X, batch_size=1024):
    model.eval()
    errors = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            b = torch.from_numpy(X[i:i+batch_size]).float().to(DEVICE)
            errors.append(weighted_mse(model(b), b).cpu().numpy())
    return np.concatenate(errors)

def ensemble_score(models, X, batch_size=1024):
    """Mean cyber-weighted reconstruction error across all ensemble members."""
    all_scores = np.stack([weighted_reconstruction_errors(m, X, batch_size) for m in models], axis=0)
    return all_scores.mean(axis=0)

benign_mask_train = (y_train_np == 0)
X_benign_train = X_train_scaled[benign_mask_train]
print(f'Benign training pool: {len(X_benign_train)} samples')


## 4. Build the k-fold ensemble (5 folds -> 5 members + pooled out-of-fold calibration)

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fold_models = []
oof_errors = np.zeros(len(X_benign_train))

for fold_idx, (train_idx, held_idx) in enumerate(kf.split(X_benign_train)):
    print(f'Fold {fold_idx+1}/{N_FOLDS}: training on {len(train_idx)}, calibrating on {len(held_idx)}...')
    torch.manual_seed(RANDOM_STATE + fold_idx)
    model = AnomalyAutoencoder(n_feat).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    X_fit = X_benign_train[train_idx]
    ds = TensorDataset(torch.from_numpy(X_fit).float(), torch.from_numpy(X_fit).float())
    loader = DataLoader(ds, batch_size=256, shuffle=True,
                         generator=torch.Generator().manual_seed(RANDOM_STATE + fold_idx))

    for epoch in range(30):
        model.train()
        for X_b, _ in loader:
            X_b = X_b.to(DEVICE)
            opt.zero_grad()
            loss = weighted_mse(model(X_b), X_b).mean()
            loss.backward()
            opt.step()

    X_held = X_benign_train[held_idx]
    oof_errors[held_idx] = weighted_reconstruction_errors(model, X_held)
    fold_models.append(model)

threshold = np.percentile(oof_errors, 95)
print(f'\nPooled out-of-fold threshold (95th percentile): {threshold:.6f}')
print(f'(NB7 baseline, single 80/20 split, mean across 5 seeds: see comparison below)')


## 5. Evaluate the hardened ensemble gate

One evaluation run (not a seed loop) -- the k-fold ensemble already
integrates variance-reduction across 5 models by construction, which was the
purpose the seed loop served for the single-AE gate in NB7.

In [ ]:
clean_scores = ensemble_score(fold_models, X_test_scaled)
clean_flagged = clean_scores > threshold
fpr_clean = clean_flagged[benign_mask].mean()
tpr_clean = clean_flagged[attack_mask].mean()
print(f'FPR (clean, test set): {fpr_clean:.4f}   TPR (clean, test set): {tpr_clean:.4f}')
print(f'(NB7 single-AE baseline: FPR 0.1571 +/- 0.0483, TPR 0.6176 +/- 0.0272)')

pgd_scores = ensemble_score(fold_models, X_adv_pgd)
pgd_flagged = pgd_scores > threshold
pgd_detection = pgd_flagged[attack_mask].mean()
std_residual_pgd = ((xgb_model.predict(X_adv_pgd)[attack_mask] == 0) & ~pgd_flagged[attack_mask]).mean()
def_residual_pgd = ((xgb_defended.predict(X_adv_pgd)[attack_mask] == 0) & ~pgd_flagged[attack_mask]).mean()
print(f'\nPGD detection: {pgd_detection:.4f}  residual(standard): {std_residual_pgd:.4f}  residual(defended): {def_residual_pgd:.4f}')
print(f'(NB7 baseline: detection 0.6594 +/- 0.0234, residual(defended) 0.0385 +/- 0.0016)')

ppo_unc_scores = ensemble_score(fold_models, X_ppo_unc)
ppo_unc_flagged = ppo_unc_scores > threshold
ppo_unc_detection = ppo_unc_flagged.mean()
unc_gate_catches = ppo_unc_flagged[evaded_ppo_unc]
unc_residual = evaded_ppo_unc.mean() * (1 - (unc_gate_catches.mean() if evaded_ppo_unc.any() else 0.0))
print(f'\nPPO-unconstrained detection: {ppo_unc_detection:.4f}  residual: {unc_residual:.4f}')
print(f'(NB7 baseline: detection 0.7858 +/- 0.0134, residual 0.2142 +/- 0.0134)')

ppo_bud_scores = ensemble_score(fold_models, X_ppo_bud)
ppo_bud_flagged = ppo_bud_scores > threshold
ppo_bud_detection = ppo_bud_flagged.mean()
bud_gate_catches = ppo_bud_flagged[evaded_ppo_bud]
bud_residual = evaded_ppo_bud.mean() * (1 - (bud_gate_catches.mean() if evaded_ppo_bud.any() else 0.0))
print(f'\nPPO-budgeted detection: {ppo_bud_detection:.4f}  residual: {bud_residual:.4f}')
print(f'(NB7 baseline: residual 0.0568 +/- 0.0016)')

adap_scores = ensemble_score(fold_models, X_adv_adaptive)
adap_flagged = adap_scores > threshold
adap_detection = adap_flagged[attack_mask].mean()
adap_residual = ((xgb_defended.predict(X_adv_adaptive)[attack_mask] == 0) & ~adap_flagged[attack_mask]).mean()
print(f'\nAdaptive PGD detection: {adap_detection:.4f}  residual: {adap_residual:.4f}')
print(f'(NB7 baseline: residual 0.0929 +/- 0.0036, no-gate evasion this surrogate: {adaptive_evasion_defended:.4f})')

# Non-distorted PPO subset accuracy (same fix applied here as in the corrected NB7)
def final_system_predict(clf, X, flagged):
    base_pred = clf.predict(X)
    proba = clf.predict_proba(X)
    override_mask = (base_pred == 0) & flagged
    fallback = 1 + np.argmax(proba[:, 1:], axis=1)
    final_pred = base_pred.copy()
    final_pred[override_mask] = fallback[override_mask]
    return final_pred

ppo_unc_subset_acc = (final_system_predict(xgb_defended, X_ppo_unc, ppo_unc_flagged) == y_ppo_unc).mean()
ppo_bud_subset_acc = (final_system_predict(xgb_defended, X_ppo_bud, ppo_bud_flagged) == y_ppo_bud).mean()
print(f'\nPPO-unconstrained subset accuracy (override recovers true label): {ppo_unc_subset_acc:.4f}')
print(f'PPO-budgeted subset accuracy (override recovers true label):       {ppo_bud_subset_acc:.4f}')


## 6. Direct comparison: hardened ensemble gate vs. NB7's single-AE baseline

In [ ]:
# NOTE: this loads the ORIGINAL (tuned-model) single-AE gate's results as the
# comparison point -- there is no baseline-model equivalent of that simpler gate to
# compare against. The comparison below therefore mixes two changes at once (gate
# architecture: single-AE -> hardened ensemble, AND classifier: tuned -> baseline),
# not gate architecture alone. Read the 'Change' column as descriptive context, not
# as an isolated measurement of what hardening alone contributes for this pipeline.
baseline = joblib.load(f'defence2_results_{PIPELINE_VERSION}.pkl')

print('=' * 90)
print(f"{'Metric':<32} {'NB7 baseline (mean+/-std)':>28} {'NB7b hardened':>18} {'Change':>10}")
print('-' * 90)

def compare(label, baseline_key, hardened_val):
    b_mean = baseline[f'{baseline_key}_mean']
    b_std = baseline[f'{baseline_key}_std']
    print(f"{label:<32} {b_mean:>18.4f} +/- {b_std:<6.4f} {hardened_val:>18.4f} {hardened_val-b_mean:>+10.4f}")

compare('FPR (clean)', 'fpr_clean', fpr_clean)
compare('TPR (clean)', 'tpr_clean', tpr_clean)
compare('PGD detection', 'pgd_detection', pgd_detection)
compare('PGD residual (defended)', 'pgd_residual_defended', def_residual_pgd)
compare('PPO-unconstrained detection', 'ppo_unc_detection', ppo_unc_detection)
compare('PPO-unconstrained residual', 'ppo_unc_residual', unc_residual)
compare('PPO-budgeted detection', 'ppo_bud_detection', ppo_bud_detection)
compare('PPO-budgeted residual', 'ppo_bud_residual', bud_residual)
compare('Adaptive PGD detection', 'adap_detection', adap_detection)
compare('Adaptive PGD residual', 'adap_residual', adap_residual)

worst_case_baseline = max(baseline['pgd_residual_defended_mean'], baseline['ppo_unc_residual_mean'],
                          baseline['ppo_bud_residual_mean'], baseline['adap_residual_mean'])
worst_case_hardened = max(def_residual_pgd, unc_residual, bud_residual, adap_residual)
print(f"\n{'Worst-case residual evasion':<32} {worst_case_baseline:>18.4f} {'':>9} {worst_case_hardened:>18.4f} "
      f"{worst_case_hardened-worst_case_baseline:>+10.4f}")

if worst_case_hardened < worst_case_baseline:
    print(f'\n-> Hardening REDUCED worst-case residual evasion by '
          f'{(worst_case_baseline-worst_case_hardened)/worst_case_baseline*100:.1f}% relative.')
else:
    print(f'\n-> Hardening did NOT improve the worst-case scenario -- the ~4x cyber-weighting, '
          f'ensembling, and k-fold calibration were insufficient for this attack tier. This is a '
          f'legitimate negative result: the residual evasion here may need the adaptive-attack-vs-gate '
          f'approach (testing an attacker who optimizes against the gate directly) rather than further '
          f'passive hardening.')


## Save hardened gate

In [ ]:
ensemble_state = {'threshold': float(threshold), 'cyber_boost': CYBER_BOOST,
                  'cyber_idx': CYBER_IDX.tolist(), 'n_folds': N_FOLDS,
                  'members': [m.state_dict() for m in fold_models]}
torch.save(ensemble_state, f'autoencoder_gate_hardened_baseline_{PIPELINE_VERSION}.pt')

joblib.dump({
    'PIPELINE_VERSION': PIPELINE_VERSION, 'cyber_boost': CYBER_BOOST, 'n_folds': N_FOLDS,
    'threshold': float(threshold),
    'fpr_clean': float(fpr_clean), 'tpr_clean': float(tpr_clean),
    'pgd_detection': float(pgd_detection), 'pgd_residual_standard': float(std_residual_pgd),
    'pgd_residual_defended': float(def_residual_pgd),
    'ppo_unc_detection': float(ppo_unc_detection), 'ppo_unc_residual': float(unc_residual),
    'ppo_bud_detection': float(ppo_bud_detection), 'ppo_bud_residual': float(bud_residual),
    'adap_detection': float(adap_detection), 'adap_residual': float(adap_residual),
    'ppo_unc_subset_acc': float(ppo_unc_subset_acc), 'ppo_bud_subset_acc': float(ppo_bud_subset_acc),
}, f'defence2b_results_baseline_{PIPELINE_VERSION}.pkl')

print(f'NB7b-baseline ({PIPELINE_VERSION}, multiclass, UNTUNED XGBoost) artifacts saved:')
for f in [f'autoencoder_gate_hardened_baseline_{PIPELINE_VERSION}.pt', f'defence2b_results_baseline_{PIPELINE_VERSION}.pkl']:
    if os.path.exists(f):
        print(f'  {f}  ({os.path.getsize(f)/1e6:.2f} MB)')


## 7. Reproducibility check: does the worst-case improvement hold across independent builds?

The single k-fold run above showed PPO-unconstrained residual improving from
0.2142 (NB7 baseline) to 0.1712 (hardened) -- a 20% relative reduction. Before
treating that as a locked result, this reruns the **entire k-fold ensemble
construction** at 2 more outer random states, checking whether the
improvement is consistent across independent builds or was a favorable
single run.

In [ ]:
OUTER_SEEDS_CHECK = [7, 123]  # in addition to the RANDOM_STATE=42 build already done above
repro_results = []

repro_results.append({
    'outer_seed': RANDOM_STATE, 'fpr_clean': fpr_clean, 'tpr_clean': tpr_clean,
    'pgd_residual_defended': def_residual_pgd, 'ppo_unc_residual': unc_residual,
    'ppo_bud_residual': bud_residual, 'adap_residual': adap_residual,
})

for outer_seed in OUTER_SEEDS_CHECK:
    print(f'\n{"="*60}\nRebuilding ensemble with outer_seed={outer_seed}\n{"="*60}')
    kf_r = KFold(n_splits=N_FOLDS, shuffle=True, random_state=outer_seed)
    fold_models_r = []
    oof_errors_r = np.zeros(len(X_benign_train))

    for fold_idx, (train_idx, held_idx) in enumerate(kf_r.split(X_benign_train)):
        torch.manual_seed(outer_seed * 100 + fold_idx)
        model_r = AnomalyAutoencoder(n_feat).to(DEVICE)
        opt_r = torch.optim.Adam(model_r.parameters(), lr=1e-3)
        X_fit_r = X_benign_train[train_idx]
        ds_r = TensorDataset(torch.from_numpy(X_fit_r).float(), torch.from_numpy(X_fit_r).float())
        loader_r = DataLoader(ds_r, batch_size=256, shuffle=True,
                               generator=torch.Generator().manual_seed(outer_seed * 100 + fold_idx))
        for epoch in range(30):
            model_r.train()
            for X_b, _ in loader_r:
                X_b = X_b.to(DEVICE)
                opt_r.zero_grad()
                loss = weighted_mse(model_r(X_b), X_b).mean()
                loss.backward()
                opt_r.step()
        X_held_r = X_benign_train[held_idx]
        oof_errors_r[held_idx] = weighted_reconstruction_errors(model_r, X_held_r)
        fold_models_r.append(model_r)

    threshold_r = np.percentile(oof_errors_r, 95)

    clean_scores_r = ensemble_score(fold_models_r, X_test_scaled)
    clean_flagged_r = clean_scores_r > threshold_r
    fpr_r = clean_flagged_r[benign_mask].mean()
    tpr_r = clean_flagged_r[attack_mask].mean()

    pgd_flagged_r = ensemble_score(fold_models_r, X_adv_pgd) > threshold_r
    def_res_r = ((xgb_defended.predict(X_adv_pgd)[attack_mask] == 0) & ~pgd_flagged_r[attack_mask]).mean()

    ppo_unc_flagged_r = ensemble_score(fold_models_r, X_ppo_unc) > threshold_r
    unc_catch_r = ppo_unc_flagged_r[evaded_ppo_unc]
    unc_res_r = evaded_ppo_unc.mean() * (1 - (unc_catch_r.mean() if evaded_ppo_unc.any() else 0.0))

    ppo_bud_flagged_r = ensemble_score(fold_models_r, X_ppo_bud) > threshold_r
    bud_catch_r = ppo_bud_flagged_r[evaded_ppo_bud]
    bud_res_r = evaded_ppo_bud.mean() * (1 - (bud_catch_r.mean() if evaded_ppo_bud.any() else 0.0))

    adap_flagged_r = ensemble_score(fold_models_r, X_adv_adaptive) > threshold_r
    adap_res_r = ((xgb_defended.predict(X_adv_adaptive)[attack_mask] == 0) & ~adap_flagged_r[attack_mask]).mean()

    repro_results.append({
        'outer_seed': outer_seed, 'fpr_clean': fpr_r, 'tpr_clean': tpr_r,
        'pgd_residual_defended': def_res_r, 'ppo_unc_residual': unc_res_r,
        'ppo_bud_residual': bud_res_r, 'adap_residual': adap_res_r,
    })
    print(f'  FPR: {fpr_r:.4f}  TPR: {tpr_r:.4f}  PGD res: {def_res_r:.4f}  '
          f'PPO-unc res: {unc_res_r:.4f}  PPO-bud res: {bud_res_r:.4f}  Adaptive res: {adap_res_r:.4f}')


## Reproducibility verdict

In [ ]:
print('=' * 80)
print('REPRODUCIBILITY CHECK: hardened gate across 3 independent k-fold builds (multiclass)')
print('=' * 80)
print(f"{'outer_seed':<12} {'FPR':>8} {'TPR':>8} {'PGD_res':>9} {'PPO_unc_res':>12} {'PPO_bud_res':>12} {'Adap_res':>9}")
for r in repro_results:
    print(f"{r['outer_seed']:<12} {r['fpr_clean']:>8.4f} {r['tpr_clean']:>8.4f} {r['pgd_residual_defended']:>9.4f} "
          f"{r['ppo_unc_residual']:>12.4f} {r['ppo_bud_residual']:>12.4f} {r['adap_residual']:>9.4f}")

unc_vals = [r['ppo_unc_residual'] for r in repro_results]
print(f"\nPPO-unconstrained residual across builds: mean={np.mean(unc_vals):.4f}, std={np.std(unc_vals):.4f}, "
      f"range=[{min(unc_vals):.4f}, {max(unc_vals):.4f}]")
print(f"NB7 ORIGINAL TUNED-MODEL single-AE baseline (5-seed mean, reference only): 0.2142 +/- 0.0134")

# CHANGED: the 0.2142 comparison point below is the TUNED model's single-AE gate
# result -- there is no baseline-model equivalent single-AE gate to compare against,
# so this check is kept for reference only and should NOT be read as isolating a pure
# hardening effect for the baseline pipeline (see the same caveat in the cell above).
n_improved = sum(1 for v in unc_vals if v < 0.2142)
print(f"\nHardened gate's PPO-unconstrained residual beat the ORIGINAL TUNED-MODEL "
      f"single-AE baseline (0.2142) in {n_improved}/{len(unc_vals)} independent k-fold "
      f"builds -- reference only, not an apples-to-apples hardening-effect measurement "
      f"for this baseline-model pipeline.")
if n_improved >= 2:
    print("-> The improvement is CONSISTENT across builds -- the worst-case reduction is "
          "real, not a single favorable run. Safe to report 0.1712 (or the 3-build mean) as "
          "the locked hardened-gate worst-case residual for multiclass.")
elif n_improved == 0:
    print("-> The original run's improvement does NOT reproduce -- likely a single favorable "
          "build. Report the hardened gate's worst-case more conservatively (closer to the "
          "3-build mean, with the range stated) rather than the single best-observed number.")
else:
    print("-> Mixed across builds -- report the 3-build mean and range rather than the single "
          "best-observed 0.1712 figure, to avoid overstating the improvement's reliability.")

joblib.dump(repro_results, f'defence2b_repro_check_baseline_{PIPELINE_VERSION}.pkl')
print(f'\nSaved: defence2b_repro_check_baseline_{PIPELINE_VERSION}.pkl')


In [ ]:
import joblib
data = joblib.load('phase1_data_v4.pkl')
print(data['FEATURE_COLS'])